# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the `mlcroissant` library. You'll learn to load Croissant-based metadata, inspect record sets and fields by their `@id`, and perform EDA with pandas.

### Dataset Source
The FAIR^2 dataset's schema URL:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using dataset metadata.

In [ ]:
# List all available record sets and their @ids
print("Available Record Sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")

# Select a record set to inspect further (using an example @id from the loaded sets)
if record_sets:
    example_rs = record_sets[0]
    print(f"\nFields for Record Set '@id': {example_rs.id} ({example_rs.name}):")
    for field in example_rs.fields:
        print(f"  - field @id: {field.id}, name: {getattr(field, 'name', '')}")

## 3. Data Extraction
Load data from specific record sets into DataFrames. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract all available record sets by their @id
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load records as list of dicts using record_set @id
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for record set '@id': {record_set_id}")

# Display columns for the first extracted record set (if any data is available)
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nFields (@id) in DataFrame for '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records using a numeric field, normalization, and grouping. All columns are referenced via their `@id`.

In [ ]:
# Choose one record set DataFrame for EDA
if dataframes:
    df = list(dataframes.values())[0]
    df_id = list(dataframes.keys())[0]
    print(f"Analyzing record set '@id': {df_id}")
    print("\nColumns available:\n", df.columns.tolist())
    
    # Try to find a numeric (float or int) field automatically
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field:
        # Try to coerce columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break
            except Exception:
                continue

    if numeric_field:
        print(f"\nUsing numeric field '@id': {numeric_field}")
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        norm_field = f"{numeric_field}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_field]].head())

        # Group by a non-numeric field if available
        group_field = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped mean values by '@id': {group_field}")
            display(grouped_df.head())
        else:
            print("No non-numeric field found to group by.")
    else:
        print("No numeric field detected for EDA.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the normalized numeric field
if 'filtered_df' in locals() and not filtered_df.empty and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # If grouping was done, plot group means
    if 'grouped_df' in locals() and not grouped_df.empty:
        grouped_df[numeric_field].plot(kind='bar', figsize=(10,4))
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean of {numeric_field}")
        plt.tight_layout()
        plt.show()
else:
    print("No filtered data or numeric field found for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Load and parse the FAIR^2 dataset's Croissant schema using `mlcroissant`
- Explore record sets and field `@id` values
- Extract and inspect DataFrames using record set `@id`
- Perform basic EDA including filtering, normalization, grouping, and visualization referencing fields by `@id`

Refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/) for further details on schema structure and advanced data access.
